# Behavioral Decoding from GPFA Latent Trajectories

This notebook is the **behavioral decoding** evaluation for GPFA latent variable
models, part of the broader latent-dynamics evaluation suite (see the repo
README for the full picture, including how this fits alongside leave-neuron-out
prediction error and co-smoothing).

**Question this notebook answers:** do the low-dimensional latent trajectories
GPFA extracts from population spiking actually track real behavior — or could
they just as easily be capturing structured noise?

**Approach:** for each candidate latent dimensionality, fit GPFA on training
trials, transform held-out test trials into latent space, and train a linear
(Ridge) decoder to predict a behavioral variable from the latents. Decoding
accuracy (cross-validated R²) as a function of dimensionality tells us both
(a) whether the latents are behaviorally meaningful at all, and (b) how many
latent dimensions are needed to capture that behavioral signal.

**Sections**

1. Setup and data loading (mirrors the main GPFA analysis notebook)
2. Bin spike trains for GPFA
3. Single-behavior decoding walkthrough: lick rate
4. Multi-behavior decoding: sweep dimensionality across all measured behaviors

## 1. Setup

Import dependencies and add the project root to the path so the local
`Analysis_library` package and `data_paths` module can be imported.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import neo
import quantities as pq
from elephant.gpfa import GPFA

notebook_dir = Path.cwd()
project_root = notebook_dir.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from data_paths import DATA_PATH
from Analysis_library.file_loading import SessionData
import Analysis_library.analysis as ana_lib

## 2. Load Session Data, Select Units, Build GPFA Trials

This mirrors the data-preparation steps from the main GPFA analysis notebook:

- Load the session for `SESSION_KEY` and set binning/trial-window parameters
- Select well-isolated ("good") units and drop units with fewer than
  `MIN_SPIKES` total spikes
- Convert each unit's spike times into per-trial `neo.SpikeTrain` objects
  aligned to trial onset (`gpfa_data`)

See the main notebook for a more detailed breakdown of these steps.

In [ ]:
SESSION_KEY = "332"
sessions = [SessionData(DATA_PATH[SESSION_KEY])]
s = sessions[0]

print("Session:", s.mat_file)
print("Number of units:", len(s.units_dict["spikes"]))
print("Number of trials:", len(s.trials_dict["t"]))

SAMPLE_RATE_HZ = 30000.0
BIN_SIZE_MS = 30 # Consider using a smaller bin size (e.g., 10 ms) for better temporal resolution, but this may increase computational load.
PRE_TRIAL_S = 1.0
POST_TRIAL_S = 4.0

good_units = ana_lib.get_good_units(s.units_dict, verbose=False)
good_units = np.asarray(good_units, dtype=int)
print("Number of good units:", len(good_units))
trial_times = np.asarray(s.trials_dict["t"], dtype=float)

print("\nFirst 3 trial times:")
print(trial_times[:3])
# shape: trial x neuron x time bins

spikes = s.units_dict["spikes"]
TRIAL_DURATION_S = PRE_TRIAL_S + POST_TRIAL_S

# Count spikes for each unit across all trial windows
spike_counts_per_unit = []

for unit_idx in good_units:
    total_spikes = 0

    for trial_time in trial_times:
        trial_start = trial_time - PRE_TRIAL_S
        trial_stop = trial_time + POST_TRIAL_S
        spike_samples = np.asarray(spikes[unit_idx], dtype=float).squeeze()
        spike_times_s = (spike_samples / SAMPLE_RATE_HZ)
        trial_spikes = spike_times_s[(spike_times_s >= trial_start) & (spike_times_s < trial_stop)]
        total_spikes += len(trial_spikes)
    spike_counts_per_unit.append(total_spikes)
spike_counts_per_unit = np.asarray(spike_counts_per_unit)

# Keep units with at least 5 spikes across trial windows
MIN_SPIKES = 5
gpfa_units = good_units[spike_counts_per_unit >= MIN_SPIKES]

print("Original good units:", len(good_units))
print("Units kept for GPFA:", len(gpfa_units))

# Build GPFA trials

gpfa_data = []

for trial_time in trial_times:
    trial_spiketrains = []
    trial_start = trial_time - PRE_TRIAL_S
    trial_stop = trial_time + POST_TRIAL_S

    for unit_idx in gpfa_units:
        spike_samples = np.asarray(spikes[unit_idx], dtype=float).squeeze()
        spike_times_s = (spike_samples / SAMPLE_RATE_HZ)
        trial_spikes = spike_times_s[(spike_times_s >= trial_start) & (spike_times_s < trial_stop)]
        trial_spikes = (trial_spikes - trial_start)
        spike_train = neo.SpikeTrain(
            trial_spikes * pq.s,
            t_start=0 * pq.s,
            t_stop=TRIAL_DURATION_S * pq.s
        )
        trial_spiketrains.append(spike_train)
    
    gpfa_data.append(trial_spiketrains)

print("Number of neurons per trial:", len(gpfa_data[0]))
print(s.data.keys())

## 3. Bin Spike Trains

Set the candidate latent dimensionalities to sweep over (`X_DIMS`) and convert
each trial's spike trains into binned spike counts
(`trials x neurons x time bins`) at `BIN_SIZE_MS` resolution.

In [ ]:
from sklearn.model_selection import KFold
from elephant.conversion import BinnedSpikeTrain
import numpy as np
import time

X_DIMS = [1, 2, 3, 4, 5, 8, 10, 15, 20] # [2, 4, 6, 8]
N_FOLDS = 4
RANDOM_SEED = 42

binned_trials = []

# trials x neurons x time bins
for trial in gpfa_data:
    binned = BinnedSpikeTrain(trial, bin_size=BIN_SIZE_MS * pq.ms)
    counts = binned.to_array()
    binned_trials.append(counts)
binned_trials = np.asarray(binned_trials)

print( "Binned data shape:", binned_trials.shape)

## 4. Single-Behavior Decoding Walkthrough: Lick Rate

Before sweeping over every behavioral variable, this section walks through the
full decoding procedure for one behavior — lick rate — as a worked example.

### 4.1 Bin lick rate into GPFA time bins

The raw lick-rate trace is sampled on its own clock, so it needs to be averaged
into the same per-trial, per-bin structure as `binned_trials` before it can be
compared to the latents.

In [ ]:
# Behavioral decoding from GPFA latents: lick rate prediction using Ridge regression and cross-validation

from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

# Bin lick rate into GPFA trial bins: behavioral signal must be converted into the same trial-by-trial binning for direct comparison
lick_rate = np.asarray(s.data["behavior"]["lick_rate"]).squeeze()
behavior_time = np.asarray(s.data["t"]).squeeze()  # renamed from `time` to avoid shadowing the `time` module imported above
binned_lick = []

for trial_time in trial_times:
    trial_start = trial_time - PRE_TRIAL_S
    trial_bins = []

    # Average lick rate within each GPFA time bin
    for b in range(binned_trials.shape[2]):
        bin_start = trial_start + b * BIN_SIZE_MS / 1000
        bin_stop = bin_start + BIN_SIZE_MS / 1000
        idx = ((behavior_time >= bin_start) & (behavior_time < bin_stop))
        if np.any(idx):
            trial_bins.append(np.mean(lick_rate[idx]))
        else:
            trial_bins.append(np.nan)
    binned_lick.append(trial_bins)
binned_lick = np.asarray(binned_lick)
print("Binned lick shape:", binned_lick.shape)

### 4.2 Cross-validated decoding across latent dimensionalities

For each candidate `x_dim`:

1. Split trials into train/test folds (`N_DECODE_FOLDS`-fold CV over trials)
2. Fit GPFA **only** on the training trials for that fold
3. Transform both train and test trials with that same fitted model
4. Flatten latents and lick-rate bins across trials into `(time bins, x_dim)` /
   `(time bins,)` arrays, dropping bins with missing behavioral data
5. Fit a Ridge regression decoder on the training latents and score it
   (R²) on the held-out test latents

This gives a cross-validated R² per fold, per candidate dimensionality —
never scoring the decoder on trials (or a GPFA fit) it has seen during training.

In [ ]:
def make_trial_split(n_trials, n_folds=4, seed=RANDOM_SEED):
    """Train/test split by holding out whole trials."""
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return list(kf.split(np.arange(n_trials)))

N_DECODE_FOLDS = 5
trial_splits = make_trial_split(len(gpfa_data), n_folds=N_DECODE_FOLDS, seed=RANDOM_SEED)

# Decode behavior from GPFA latents
behavior_scores = {}
for x_dim in X_DIMS:
    print(f"\nTesting x_dim = {x_dim}")
    fold_scores = []
    # Cross-validated decoding of lick rate from latent trajectories
    for train_idx, test_idx in trial_splits:
        # Fit GPFA using ONLY training trials -- test trials never seen during fitting
        train_trials = [gpfa_data[i] for i in train_idx]
        test_trials = [gpfa_data[i] for i in test_idx]
        gpfa = GPFA(
            bin_size=BIN_SIZE_MS * pq.ms,
            x_dim=x_dim,
            em_max_iters=100,
            verbose=False
        )
        gpfa.fit(train_trials)
        
        # Transform each split separately using the SAME fitted model
        latent_train = gpfa.transform(train_trials)
        latent_test = gpfa.transform(test_trials)

        # Concatenate all training trials into one dataset, each row corresponds to one time bin
        X_train = np.concatenate([lt.T for lt in latent_train], axis=0)
        y_train = np.concatenate([binned_lick[i] for i in train_idx])

        # Build held-out dataset, decoder never sees the held-out trials during evaluation
        X_test = np.concatenate([lt.T for lt in latent_test], axis=0)
        y_test = np.concatenate([binned_lick[i] for i in test_idx])

        # Remove bins with missing behavioral measurements
        valid_train = ~np.isnan(y_train)
        valid_test = ~np.isnan(y_test)
        X_train = X_train[valid_train]
        y_train = y_train[valid_train]
        X_test = X_test[valid_test]
        y_test = y_test[valid_test]

        # Train linear decoder: ridge regression predicts lick rate from the GPFA latent variables while applying L2 regularization to reduce overfitting
        decoder = Ridge(alpha=1.0)
        decoder.fit(X_train, y_train)
        prediction = decoder.predict(X_test)

        # Measure decoding performance using coefficient of determination: higher R² indicates the latent trajectories better explain behavior
        score = r2_score(y_test, prediction)
        fold_scores.append(score)
    behavior_scores[x_dim] = fold_scores
    print("Mean R²:", np.mean(fold_scores))

### 4.3 Summarize decoding performance vs. dimensionality

Plot mean ± SEM cross-validated R² across folds for each candidate `x_dim`, and
report the best-performing dimensionality for decoding lick rate.

In [ ]:
from scipy.stats import sem

means = [np.mean(behavior_scores[d]) for d in X_DIMS]
sems  = [sem(behavior_scores[d]) for d in X_DIMS]

# Find best latent dimensionality
best_idx = np.argmax(means)
best_x_dim = X_DIMS[best_idx]
best_r2 = means[best_idx]

print("Behavior decoding performance:")
for d, mean_r2, sem_r2 in zip(X_DIMS, means, sems):
    print(
        f"x_dim = {d}: "
        f"mean CV R² = {mean_r2:.4f} ± {sem_r2:.4f}"
    )

print("Best GPFA dimensionality:")
print(f"  x_dim = {best_x_dim}")
print(f"  mean CV R² = {best_r2:.4f}")

plt.figure(figsize=(6,4))
plt.errorbar(X_DIMS, means, yerr=sems, fmt="o-", capsize=4)
plt.axvline(best_x_dim, linestyle="--", label=f"Best x_dim={best_x_dim}")
plt.xlabel("Latent dimensionality")
plt.ylabel("Cross-validated $R^2$")
plt.title("Behavioral decoding from GPFA latents")
plt.grid(True)
plt.legend()

plt.show()

## 5. Multi-Behavior Decoding Sweep

Repeat the same cross-validated decoding procedure for every measured
behavioral variable at once (lick rate, wheel velocity, whisking, pupil
area/radius, respiration, face motion, and respiration frequency/amplitude),
so dimensionality selection and behavioral relevance can be compared side by
side across behaviors.

For each `x_dim`, GPFA is fit **once per cross-validation fold** (not once per
behavior) — the same fold's fitted model and latents are reused to decode every
behavior, which keeps this section reasonably efficient given how many
behaviors are being swept.

> **Note:** this cell can take a while to run — it fits
> `len(X_DIMS) * n_folds` GPFA models, each of which is then used to decode
> every behavioral variable.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

behavior_signals = {
    "Lick rate": s.data["behavior"]["lick_rate"],
    "Wheel": s.data["physiology"]["wheel"],
    "Whisking": s.data["physiology"]["whisking"],
    "Pupil area": s.data["physiology"]["pupil_area"],
    "Pupil radius": s.data["physiology"]["pupil_radius"],
    "Respiration": s.data["physiology"]["respiration"],
    "Face motion": s.data["physiology"]["facexor"],
    "Respiration frequency": s.data["physiology"]["resp_stats"]["inst_resp_f"],
    "Respiration amplitude": s.data["physiology"]["resp_stats"]["inst_resp_amp"],
}

behavior_time = np.asarray(s.data["t"]).squeeze()  # renamed from `time` to avoid shadowing the `time` module imported above
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# Bin every behavioral signal once
binned_behaviors = {}
for name, signal in behavior_signals.items():
    signal = np.asarray(signal).squeeze()
    if signal.ndim != 1:
        continue
    trials = []
    for trial_time in trial_times:
        trial_start = trial_time - PRE_TRIAL_S
        bins = []
        for b in range(binned_trials.shape[2]):
            bin_start = trial_start + b * BIN_SIZE_MS / 1000
            bin_stop = bin_start + BIN_SIZE_MS / 1000
            idx = (behavior_time >= bin_start) & (behavior_time < bin_stop)
            bins.append(np.mean(signal[idx]) if np.any(idx) else np.nan)
        trials.append(bins)
    binned_behaviors[name] = np.asarray(trials)

# Fit GPFA ONCE for each latent dimensionality
all_results = {}

for x_dim in X_DIMS:
    print(f"\nFitting GPFA: x_dim = {x_dim}")
    all_results[x_dim] = {name: [] for name in binned_behaviors.keys()}

    gpfa_data_arr = np.asarray(gpfa_data, dtype=object)  # allows fancy indexing by trial

    for fold_i, (train_idx, test_idx) in enumerate(kf.split(gpfa_data_arr)):
        # Fit GPFA ONLY on training trials for this fold
        gpfa = GPFA(
            bin_size=BIN_SIZE_MS * pq.ms,
            x_dim=x_dim,
            em_max_iters=100,
            verbose=False
        )
        train_spiketrains = [gpfa_data[i] for i in train_idx]
        gpfa.fit(train_spiketrains)

        # Transform train and test trials separately using the train-fit model
        train_latents = gpfa.transform([gpfa_data[i] for i in train_idx])
        test_latents = gpfa.transform([gpfa_data[i] for i in test_idx])

        # Now decode each behavior using only this fold's train/test latents
        for behavior_name, binned_behavior in binned_behaviors.items():
            X_train = np.concatenate([lat.T for lat in train_latents])
            y_train = np.concatenate([binned_behavior[i] for i in train_idx])
            X_test = np.concatenate([lat.T for lat in test_latents])
            y_test = np.concatenate([binned_behavior[i] for i in test_idx])

            keep_train = ~np.isnan(y_train)
            keep_test = ~np.isnan(y_test)
            X_train, y_train = X_train[keep_train], y_train[keep_train]
            X_test, y_test = X_test[keep_test], y_test[keep_test]

            decoder = Ridge(alpha=1.0)
            decoder.fit(X_train, y_train)
            prediction = decoder.predict(X_test)
            all_results[x_dim][behavior_name].append(r2_score(y_test, prediction))

# Plot results
for behavior_name in binned_behaviors.keys():
    means = [np.mean(all_results[d][behavior_name]) for d in X_DIMS]
    sems = [np.std(all_results[d][behavior_name], ddof=1) / np.sqrt(len(all_results[d][behavior_name])) for d in X_DIMS]
    best_dim = X_DIMS[np.argmax(means)]

    print(f"\n{behavior_name}")
    print(f"Best x_dim = {best_dim}")
    print(f"Best R² = {np.max(means):.4f}")
    plt.figure(figsize=(6,4))
    plt.errorbar(X_DIMS, means, yerr=sems, marker="o", capsize=4)
    plt.xlabel("Latent dimensionality")
    plt.ylabel("Cross-validated $R^2$")
    plt.title(behavior_name)
    plt.grid(True)
    plt.show()

### 5.1 Summary plot: all behaviors, one figure

Overlay cross-validated R² vs. latent dimensionality for every behavior on a
single plot, to compare which behaviors are best explained by the latent
trajectories and at what dimensionality decoding performance saturates.

In [ ]:
# summary cloud plot for all behaviors
plt.figure(figsize=(10,6))

for behavior_name in binned_behaviors.keys():
    means = np.array([np.mean(all_results[d][behavior_name]) for d in X_DIMS])
    sems = np.array([np.std( all_results[d][behavior_name], ddof=1) / np.sqrt(len(all_results[d][behavior_name])) for d in X_DIMS])
    plt.plot(X_DIMS, means, marker="o", linewidth=2, label=behavior_name)
    plt.fill_between(X_DIMS, means - sems, means + sems, alpha=0.15)
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("Latent dimensionality (x_dim)")
plt.ylabel("Cross-validated R²")
plt.title("Behavior decoding from GPFA latent trajectories")
plt.legend(bbox_to_anchor=(1.05,1), loc="upper left")
plt.grid(True)
plt.tight_layout()
plt.show()